In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error

# ============================================
# LOAD DATASET
# ============================================

df = sns.load_dataset('tips')

# ============================================
# DEFINE X AND y
# ============================================

y = df['total_bill']
X = df.drop('total_bill', axis=1)

# ============================================
# TRAIN TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================================
# LABEL ENCODING
# ============================================

le1, le2, le3 = LabelEncoder(), LabelEncoder(), LabelEncoder()

X_train['sex'] = le1.fit_transform(X_train['sex'])
X_train['smoker'] = le2.fit_transform(X_train['smoker'])
X_train['time'] = le3.fit_transform(X_train['time'])

X_test['sex'] = le1.transform(X_test['sex'])
X_test['smoker'] = le2.transform(X_test['smoker'])
X_test['time'] = le3.transform(X_test['time'])

# ============================================
# ONE HOT ENCODING
# ============================================

X_train = pd.get_dummies(X_train, columns=['day'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['day'], drop_first=True)

# fix mismatch
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# ============================================
# SCALING
# ============================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================
# GRID SEARCH FOR SVR
# ============================================

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.1, 0.01],
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(
    estimator=SVR(),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1
)

# train with grid search
grid.fit(X_train, y_train)

# ============================================
# BEST MODEL
# ============================================

best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)

# ============================================
# PREDICTION
# ============================================

y_pred = best_model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2 Score:", r2)
print("RMSE:", rmse)

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params: {'C': 100, 'gamma': 0.01, 'kernel': 'rbf'}
R2 Score: 0.6145125974172526
RMSE: 5.717072290273526
